In [1]:
import xgboost as xgb

from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
from sklearn.metrics import mean_squared_error

import os

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import site_archive_nci

from pyearthtools.data.time import Petdt
from pyearthtools.pipeline.operations.xarray.join import GeospatialTimeSeriesMerge

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd

In [2]:
himawari = petdata.archive.Himawari('hourly_integral_of_surface_global_irradiance')

In [3]:
# Use pyearthtools data archive to access the data of interest
variables_of_interest = ['hus850','hus500','psl','tas']
frequency = '1hr'
domain_id = 'AUST-04'
barra = petdata.archive.BARRA_V2(
    variables_of_interest,
    frequency=frequency,
    domain_id=domain_id
)

In [4]:
barra_date = barra['2024-01']

himawar_date = himawari['2024-01']

/scratch/nf33/cd3022/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  full_ds = xr.open_mfdataset(
/scratch/nf33/cd3022/PyEarthTools/packages/data/src/pyearthtools/data/operations/index_routines.py:326: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitl

In [5]:
syd_bar = barra_date.sel( # Sydney coordinates
    latitude=-33.86,
    longitude=151.21,
    method='nearest'
).isel(height=0)
syd_him = himawar_date.sel( # Sydney coordinates
    latitude=-33.86,
    longitude=151.21,
    method='nearest'
)

In [6]:
dfs = []

# make DFs for pressure level vars
for p in syd_bar.pressure.values:
    tmp = syd_bar.sel(pressure=p).to_dataframe()
    tmp = tmp.rename(columns={'hus': f'hus{int(p)}'})
    dfs.append(tmp[['hus' + str(int(p))]])

# Add surface level vars
base = syd_bar[['psl', 'tas']].to_dataframe()

df_bar = base.join(dfs)
df_bar = df_bar.drop(columns=['latitude', 'longitude', 'crs', 'height'], errors='ignore')

df_him = syd_him[['surface_global_irradiance', 'solar_elevation']].to_dataframe()
df_him = df_him.drop(columns=['latitude', 'longitude'], errors='ignore')

data = df_bar.join(df_him, how='inner')


data['surface_global_irradiance_t1'] = data['surface_global_irradiance'].shift(-1)

data = data.dropna()

In [7]:
X = data.drop(columns=['surface_global_irradiance_t1'])
y = data['surface_global_irradiance_t1']

In [8]:
split_index = int(len(data) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

# # Convert back to DataFrame (keeps column names)
# X_scaled = pd.DataFrame(X_scaled, columns=data.columns, index=data.index)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
# Train a simple XGBoost model
model = xgb.XGBRegressor(random_state=42)  # Use any integer seed
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Quick evaluation of model performance using correlation and RMSE
correlation = np.corrcoef(y_test, y_pred)[0, 1]
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Correlation between predicted and actual values: {correlation:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")

Correlation between predicted and actual values: 0.726
Root Mean Squared Error (RMSE): 190.297


In [11]:
importance = model.feature_importances_

feat_importance = pd.Series(importance, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

print(feat_importance)

surface_global_irradiance    0.799400
solar_elevation              0.079655
hus500                       0.047277
tas                          0.034069
hus850                       0.021380
psl                          0.018219
dtype: float32
